# Preprocessing V2 - Improved Pipeline\n\nFixes from the original preprocessing:\n1. **NO forced square resize** - preserve aspect ratio, let models handle resizing\n2. **Preserve segmentation masks** - keep polygon annotations for instance segmentation\n3. **Stratified split** - balanced class distribution across train/val/test\n4. **YOLO-seg format** - polygon-based labels for YOLOv8-seg\n5. **COCO format** - per-split annotation files for Mask R-CNN

In [ ]:
import json
import os
import shutil
from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
ARCHIVE_DIR = os.path.join(BASE_DIR, "archive")
DATA_DIR = os.path.join(ARCHIVE_DIR, "data")
INPUT_JSON = os.path.join(DATA_DIR, "annotations.json")

OUTPUT_DIR = os.path.join(ARCHIVE_DIR, "dataset_v2")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Input annotations:", INPUT_JSON)

## Step 1: Load Dataset & Merge Categories (60 → 9)

In [ ]:
# =========================
# CATEGORY MAPPING (60 → 9 classes)
# =========================

category_mapping = {
    # Cigarette
    "Cigarette": "Cigarette",
    
    # Unlabeled
    "Unlabeled litter": "Unlabeled",
    
    # Plastic (largest group)
    "Plastic film": "Plastic",
    "Other plastic": "Plastic",
    "Other plastic wrapper": "Plastic",
    "Plastic bottle cap": "Plastic",
    "Plastic straw": "Plastic",
    "Polypropylene bag": "Plastic",
    "Plastic glooves": "Plastic",
    "Tupperware": "Plastic",
    "Disposable plastic cup": "Plastic",
    "Other plastic bottle": "Plastic",
    "Other plastic container": "Plastic",
    "Plastic lid": "Plastic",
    "Plastic utensils": "Plastic",
    "Single-use carrier bag": "Plastic",
    "Garbage bag": "Plastic",
    "Crisp packet": "Plastic",
    
    # Bottle
    "Clear plastic bottle": "Bottle",
    
    # Metal
    "Drink can": "Metal",
    "Metal lid": "Metal",
    "Aerosol": "Metal",
    "Aluminium blister pack": "Metal",
    "Aluminium foil": "Metal",
    "Food Can": "Metal",
    "Metal bottle cap": "Metal",
    "Pop tab": "Metal",
    "Scrap metal": "Metal",
    
    # Glass
    "Broken glass": "Glass",
    "Glass cup": "Glass",
    "Glass jar": "Glass",
    "Glass bottle": "Glass",
    
    # Paper
    "Pizza box": "Paper",
    "Egg carton": "Paper",
    "Magazine paper": "Paper",
    "Wrapping paper": "Paper",
    "Toilet tube": "Paper",
    "Paper straw": "Paper",
    "Corrugated carton": "Paper",
    "Meal carton": "Paper",
    "Other carton": "Paper",
    "Paper bag": "Paper",
    "Paper cup": "Paper",
    "Normal paper": "Paper",
    "Tissues": "Paper",
    "Drink carton": "Paper",
    "Plastified paper bag": "Paper",
    
    # Foam
    "Foam cup": "Foam",
    "Foam food container": "Foam",
    "Styrofoam piece": "Foam",
    
    # Other (rare items)
    "Food waste": "Other",
    "Carded blister pack": "Other",
    "Battery": "Other",
    "Other plastic cup": "Other",
    "Shoe": "Other",
    "Six pack rings": "Other",
    "Spread tub": "Other",
    "Squeezable tube": "Other",
    "Rope & strings": "Other",
    "Disposable food container": "Other",
}

# =========================
# LOAD DATASET
# =========================

with open(INPUT_JSON) as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

print(f"Loaded: {len(images)} images, {len(annotations)} annotations, {len(categories)} categories")

# Build old category lookup
old_cat_lookup = {cat["id"]: cat["name"] for cat in categories}

# =========================
# CREATE NEW CATEGORIES
# =========================

new_class_names = sorted(set(category_mapping.values()) | {"Other"})
new_cat_id_map = {name: idx + 1 for idx, name in enumerate(new_class_names)}
new_categories = [{"id": idx + 1, "name": name} for idx, name in enumerate(new_class_names)]

print(f"\nNew categories ({len(new_categories)}):")
for cat in new_categories:
    print(f"  {cat['id']}: {cat['name']}")

# =========================
# UPDATE ANNOTATIONS
# =========================

for ann in annotations:
    old_name = old_cat_lookup[ann["category_id"]]
    new_name = category_mapping.get(old_name, "Other")
    ann["category_id"] = new_cat_id_map[new_name]

# Verify distribution
counts = Counter(ann["category_id"] for ann in annotations)
print("\nClass distribution:")
for cat in new_categories:
    print(f"  {cat['name']}: {counts.get(cat['id'], 0)}")

## Step 2: Stratified Train/Val/Test Split (70/20/10)\n\nAssign each image to its **rarest** category for stratification. This ensures minority classes are represented in all splits.

In [ ]:
# =========================
# GROUP ANNOTATIONS BY IMAGE
# =========================

ann_by_image = defaultdict(list)
for ann in annotations:
    ann_by_image[ann["image_id"]].append(ann)

# =========================
# ASSIGN STRATIFICATION LABEL PER IMAGE
# =========================
# For multi-label images, use the rarest class for stratification
# This ensures minority classes appear in all splits

class_counts = Counter(ann["category_id"] for ann in annotations)

image_ids = []
strat_labels = []

for img in images:
    img_id = img["id"]
    img_anns = ann_by_image.get(img_id, [])
    
    if not img_anns:
        # Images with no annotations - assign to "Other"
        strat_label = new_cat_id_map["Other"]
    else:
        # Pick the rarest class in this image
        cats_in_image = [a["category_id"] for a in img_anns]
        strat_label = min(cats_in_image, key=lambda c: class_counts[c])
    
    image_ids.append(img_id)
    strat_labels.append(strat_label)

image_ids = np.array(image_ids)
strat_labels = np.array(strat_labels)

print(f"Total images for splitting: {len(image_ids)}")
print(f"Stratification label distribution:")
for cat in new_categories:
    count = np.sum(strat_labels == cat["id"])
    print(f"  {cat['name']}: {count} images")

# =========================
# STRATIFIED SPLIT: 70% train, 20% val, 10% test
# =========================

# First split: 90% train+val vs 10% test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
trainval_idx, test_idx = next(sss1.split(image_ids, strat_labels))

# Second split: from train+val, 77.8% train vs 22.2% val (= 70/20 of total)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.222, random_state=42)
train_idx, val_idx = next(sss2.split(image_ids[trainval_idx], strat_labels[trainval_idx]))

# Map back to actual indices
train_ids = set(image_ids[trainval_idx[train_idx]])
val_ids = set(image_ids[trainval_idx[val_idx]])
test_ids = set(image_ids[test_idx])

print(f"\nSplit sizes:")
print(f"  Train: {len(train_ids)} images")
print(f"  Val:   {len(val_ids)} images")
print(f"  Test:  {len(test_ids)} images")

# Verify no overlap
assert len(train_ids & val_ids) == 0, "Train/Val overlap!"
assert len(train_ids & test_ids) == 0, "Train/Test overlap!"
assert len(val_ids & test_ids) == 0, "Val/Test overlap!"
print("No overlap between splits.")

## Step 3: Copy Images to Split Folders (Original Resolution)\n\nNo resizing! Models handle their own resizing internally:\n- YOLOv8: letterbox padding (preserves aspect ratio)\n- Mask R-CNN: FPN handles multi-scale\n- Watershed: works on original resolution

In [ ]:
# =========================
# COPY IMAGES TO SPLIT FOLDERS
# =========================

split_map = {}
for img_id in train_ids:
    split_map[img_id] = "train"
for img_id in val_ids:
    split_map[img_id] = "val"
for img_id in test_ids:
    split_map[img_id] = "test"

# Create image directories
for split in ["train", "val", "test"]:
    split_dir = os.path.join(OUTPUT_DIR, "images", split)
    os.makedirs(split_dir, exist_ok=True)

# Copy images at original resolution (NO resizing)
copied = {"train": 0, "val": 0, "test": 0}
skipped = 0

for img in images:
    img_id = img["id"]
    split = split_map.get(img_id)
    if split is None:
        continue
    
    src_path = os.path.join(DATA_DIR, img["file_name"])
    
    # Flatten filename: batch_1/000001.JPG -> batch_1_000001.JPG
    flat_name = img["file_name"].replace("/", "_").replace("\\", "_")
    dst_path = os.path.join(OUTPUT_DIR, "images", split, flat_name)
    
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        img["file_name_flat"] = flat_name  # store for later use
        copied[split] += 1
    else:
        skipped += 1

print("Images copied (original resolution, no resizing):")
for split, count in copied.items():
    print(f"  {split}: {count}")
if skipped > 0:
    print(f"  Skipped (not found): {skipped}")

## Step 4: Save COCO-format Annotations Per Split\n\nFull COCO JSON with **segmentation polygons preserved** for each split. Used by Mask R-CNN and Watershed.

In [ ]:
# =========================
# SAVE COCO ANNOTATIONS PER SPLIT
# =========================

for split, id_set in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    
    split_images = []
    split_ann_ids = set()
    
    for img in images:
        if img["id"] in id_set:
            # Use flat filename for consistency
            img_entry = dict(img)
            img_entry["file_name"] = img.get("file_name_flat", img["file_name"].replace("/", "_"))
            split_images.append(img_entry)
            split_ann_ids.add(img["id"])
    
    split_annotations = [
        ann for ann in annotations 
        if ann["image_id"] in split_ann_ids
    ]
    
    split_data = {
        "images": split_images,
        "annotations": split_annotations,
        "categories": new_categories
    }
    
    output_path = os.path.join(OUTPUT_DIR, f"{split}_annotations.json")
    with open(output_path, "w") as f:
        json.dump(split_data, f)
    
    # Count annotations per class in this split
    split_counts = Counter(a["category_id"] for a in split_annotations)
    print(f"\n{split}: {len(split_images)} images, {len(split_annotations)} annotations")
    for cat in new_categories:
        print(f"  {cat['name']}: {split_counts.get(cat['id'], 0)}")

print("\nCOCO annotation files saved with segmentation polygons preserved.")

## Step 5: Convert to YOLO-seg Format\n\nYOLO segmentation format: each `.txt` label file has one line per object:\n```\nclass_id x1 y1 x2 y2 ... xn yn\n```\nWhere polygon coordinates are normalized to [0, 1] by image width/height.\n\nNote: YOLO class IDs are 0-indexed (subtract 1 from COCO IDs).

In [ ]:
# =========================
# CONVERT TO YOLO-SEG FORMAT
# =========================

image_lookup = {img["id"]: img for img in images}

for split, id_set in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    
    label_dir = os.path.join(OUTPUT_DIR, "labels", split)
    os.makedirs(label_dir, exist_ok=True)
    
    label_count = 0
    
    for img_id in id_set:
        img_info = image_lookup[img_id]
        w = img_info["width"]
        h = img_info["height"]
        
        flat_name = img_info.get("file_name_flat", img_info["file_name"].replace("/", "_"))
        label_name = os.path.splitext(flat_name)[0] + ".txt"
        label_path = os.path.join(label_dir, label_name)
        
        img_anns = ann_by_image.get(img_id, [])
        
        with open(label_path, "w") as f:
            for ann in img_anns:
                # YOLO class ID is 0-indexed
                class_id = ann["category_id"] - 1
                
                seg = ann.get("segmentation", [])
                if not seg or not isinstance(seg, list):
                    continue
                
                # Use the largest polygon if multiple exist
                polygon = max(seg, key=len) if len(seg) > 1 else seg[0]
                
                if len(polygon) < 6:  # need at least 3 points
                    continue
                
                # Normalize polygon coordinates
                normalized = []
                for i in range(0, len(polygon), 2):
                    nx = polygon[i] / w
                    ny = polygon[i + 1] / h
                    # Clamp to [0, 1]
                    nx = max(0.0, min(1.0, nx))
                    ny = max(0.0, min(1.0, ny))
                    normalized.extend([nx, ny])
                
                coords_str = " ".join(f"{c:.6f}" for c in normalized)
                f.write(f"{class_id} {coords_str}\n")
        
        label_count += 1
    
    print(f"{split}: {label_count} label files created")

print("\nYOLO-seg labels generated.")

## Step 6: Create dataset.yaml for Ultralytics YOLOv8

In [ ]:
# =========================
# CREATE dataset.yaml FOR YOLOV8
# =========================

# Class names (0-indexed, matching YOLO label files)
class_names = {i: cat["name"] for i, cat in enumerate(new_categories)}

yaml_content = f"""# TACO Waste Segmentation Dataset
# Generated by preprocessing_v2

path: {OUTPUT_DIR.replace(os.sep, '/')}
train: images/train
val: images/val
test: images/test

# Number of classes
nc: {len(new_categories)}

# Class names (0-indexed)
names:
"""

for idx, cat in enumerate(new_categories):
    yaml_content += f"  {idx}: {cat['name']}\n"

yaml_path = os.path.join(OUTPUT_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("dataset.yaml saved to:", yaml_path)
print("\nContents:")
print(yaml_content)

## Step 7: Verify Output Structure

In [ ]:
# =========================
# VERIFY OUTPUT STRUCTURE
# =========================

print("Output directory structure:")
print(f"  {OUTPUT_DIR}/")

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * (level + 1)
    folder = os.path.basename(root)
    if level == 0:
        for d in sorted(dirs):
            print(f"{indent}{d}/")
    elif level <= 2:
        file_count = len(files)
        print(f"{indent}{folder}/ ({file_count} files)")

# Verify a sample YOLO label
print("\n--- Sample YOLO-seg label ---")
sample_label_dir = os.path.join(OUTPUT_DIR, "labels", "train")
sample_files = os.listdir(sample_label_dir)
if sample_files:
    sample_path = os.path.join(sample_label_dir, sample_files[0])
    with open(sample_path) as f:
        lines = f.readlines()
    print(f"File: {sample_files[0]}")
    print(f"Objects: {len(lines)}")
    if lines:
        parts = lines[0].strip().split()
        print(f"First line: class={parts[0]}, polygon_points={len(parts)-1} values ({(len(parts)-1)//2} xy pairs)")

# Verify COCO annotation has segmentation
print("\n--- Sample COCO annotation ---")
with open(os.path.join(OUTPUT_DIR, "train_annotations.json")) as f:
    train_data = json.load(f)
sample_ann = train_data["annotations"][0]
print(f"Keys: {list(sample_ann.keys())}")
print(f"Has segmentation: {'segmentation' in sample_ann}")
if "segmentation" in sample_ann:
    print(f"Polygon points: {len(sample_ann['segmentation'][0])} values")

print("\nPreprocessing V2 complete!")